# Regression — Predicting Attraction Rating
Compares Linear Regression, Random Forest, LightGBM, and XGBoost.

**Feature engineering note:** `User_AvgRating`, `Attraction_AvgRating`, and `City_AvgRating` are K-fold out-of-fold target-encoded (not naive leave-one-out) — see `feature_engineering.py` docstring for why plain LOO leaked here (discrete 1-5 rating + moderate group sizes made it near-invertible for tree models).

In [1]:
import pandas as pd, numpy as np
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
import lightgbm as lgb, xgboost as xgb

m = pd.read_csv('../data/cleaned/master_features.csv')
train, test = m[m.__split=='train'].copy(), m[m.__split=='test'].copy()
train.shape, test.shape

((42344, 25), (10586, 25))

In [2]:
cat_cols = ['UserContinent','UserRegion','UserCountry','AttractionType','AttractionRegion','VisitModeLabel','User_FavAttractionType']
num_cols = ['VisitYear','VisitMonth','User_AvgRating','User_TotalVisits','User_FavMonth','Attraction_AvgRating','Attraction_TotalVisits','City_AvgRating']
encoders = {}
def encode(df, fit=False):
    out = df[cat_cols+num_cols].copy()
    for c in cat_cols:
        if fit:
            le = LabelEncoder(); le.fit(out[c].astype(str)); encoders[c]=le
        le = encoders[c]
        out[c] = out[c].astype(str).map(lambda v: v if v in le.classes_ else le.classes_[0])
        out[c] = le.transform(out[c])
    return out
X_train, X_test = encode(train, fit=True), encode(test)
y_train, y_test = train.Rating.values, test.Rating.values

In [3]:
scaler = StandardScaler()
X_train_s, X_test_s = scaler.fit_transform(X_train), scaler.transform(X_test)

results = []
baseline = np.full_like(y_test, y_train.mean(), dtype=float)
results.append(('Baseline (mean)', r2_score(y_test, baseline), mean_absolute_error(y_test, baseline)))

lr = LinearRegression().fit(X_train_s, y_train)
results.append(('LinearRegression', r2_score(y_test, lr.predict(X_test_s)), mean_absolute_error(y_test, lr.predict(X_test_s))))

rf = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1).fit(X_train, y_train)
results.append(('RandomForest', r2_score(y_test, rf.predict(X_test)), mean_absolute_error(y_test, rf.predict(X_test))))

lgbm = lgb.LGBMRegressor(n_estimators=300, max_depth=8, random_state=42, verbosity=-1).fit(X_train, y_train)
results.append(('LightGBM', r2_score(y_test, lgbm.predict(X_test)), mean_absolute_error(y_test, lgbm.predict(X_test))))

xgbr = xgb.XGBRegressor(n_estimators=300, max_depth=6, random_state=42).fit(X_train, y_train)
results.append(('XGBoost', r2_score(y_test, xgbr.predict(X_test)), mean_absolute_error(y_test, xgbr.predict(X_test))))

pd.DataFrame(results, columns=['Model','R2','MAE']).sort_values('R2', ascending=False)

,Model,R2,MAE
2,RandomForest,0.162286,0.687598
3,LightGBM,0.161247,0.688775
1,LinearRegression,0.130019,0.708210
4,XGBoost,0.083707,0.717630
0,Baseline (mean),-0.000002,0.759383


## Result
RandomForest/LightGBM (~R2 0.16) are the best performers — a real, verified improvement over the v1 (demographics-only) model's R2 of 0.11, driven mostly by `Attraction_AvgRating` and `User_AvgRating`. Still modest in absolute terms: most rating variance isn't explained by anything in this dataset.